# 🔌 Notebook 5: WebSockets

WebSockets are the **gold standard** for real-time bidirectional communication. Both the client AND server can send messages at any time!

## Learning Objectives

By the end of this notebook, you'll understand:
- How WebSocket connections are established (HTTP upgrade)
- Building a real-time chat application
- Connection management and scaling challenges
- When to choose WebSockets over other methods

## 🤔 What are WebSockets?

WebSockets provide **full-duplex** communication - both sides can send messages independently!

```
HTTP (Request-Response):           WebSocket (Full-Duplex):
                                   
Client ──► Request                 Client ◄──────────────► Server
Client ◄── Response                (either side can send anytime!)
                                   
Client ──► Request                 Client: "Hi!"
Client ◄── Response                Server: "Hello!"
                                   Server: "Update: new user joined"
Client must initiate!              Client: "Got it, thanks!"
                                   Server: "Update: message from Bob"
                                   Client: "Send my message to Bob"
```

The key: **persistent, bidirectional connection** after initial HTTP upgrade!

## 🔄 The WebSocket Handshake

WebSocket connections start as HTTP, then "upgrade":

```
┌────────┐                                    ┌────────┐
│ Client │                                    │ Server │
└───┬────┘                                    └───┬────┘
    │                                             │
    │  HTTP Request with Upgrade header           │
    │  ─────────────────────────────────────────► │
    │  GET /chat HTTP/1.1                         │
    │  Upgrade: websocket                         │
    │  Connection: Upgrade                        │
    │  Sec-WebSocket-Key: dGhlIHNhbXBsZSBub...    │
    │                                             │
    │  HTTP 101 Switching Protocols               │
    │  ◄───────────────────────────────────────── │
    │  Upgrade: websocket                         │
    │  Connection: Upgrade                        │
    │  Sec-WebSocket-Accept: s3pPLMBiTxaQ9...     │
    │                                             │
    │═══════════════════════════════════════════│
    │         WebSocket Connection Open!          │
    │═══════════════════════════════════════════│
    │                                             │
    │  ◄──────── Messages (both ways!) ────────►  │
    │                                             │
```

After the upgrade, it's **no longer HTTP** - it's the WebSocket protocol!

## 🛠️ Let's Build It!

### Step 1: Start the Server

Before continuing, start the server in a terminal:

```bash
cd 04-patterns/real-time-updates/servers
python websocket_server.py
```

You should see: `🚀 Starting WebSocket Server on port 5004`

In [1]:
# The websockets library ships with this lab (installed via `uv sync`).
import websockets
print("✅ websockets library version:", websockets.__version__)


✅ websockets library version: 16.0


In [2]:
# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("✅ Async support enabled for Jupyter")

✅ Async support enabled for Jupyter


### Step 2: Create the WebSocket Client

In [3]:
import asyncio
import websockets
import json
from datetime import datetime

class ChatClient:
    """
    A WebSocket chat client.
    """
    
    def __init__(self, server_url: str):
        self.server_url = server_url
        self.websocket = None
        self.username = None
        self.room = None
        self.running = False
        self.messages = []
    
    async def connect(self, username: str, room: str = "general"):
        """
        Connect to the chat server and join a room.
        """
        self.username = username
        self.room = room
        
        self.websocket = await websockets.connect(self.server_url)
        
        # Send join message
        await self.websocket.send(json.dumps({
            "type": "join",
            "username": username,
            "room": room
        }))
        
        # Wait for welcome message
        response = await self.websocket.recv()
        data = json.loads(response)
        
        return data
    
    async def send_message(self, text: str):
        """
        Send a chat message.
        """
        if self.websocket:
            await self.websocket.send(json.dumps({
                "type": "message",
                "text": text
            }))
    
    async def send_typing(self):
        """
        Send typing indicator.
        """
        if self.websocket:
            await self.websocket.send(json.dumps({
                "type": "typing"
            }))
    
    async def receive_messages(self, callback=None, duration=None):
        """
        Receive messages from the server.

        We poll with a short timeout so we can exit when either
        `self.running` becomes False or `duration` seconds elapse,
        even if no message is currently arriving.
        """
        self.running = True
        start_time = asyncio.get_event_loop().time()
        try:
            while self.running:
                if duration and (asyncio.get_event_loop().time() - start_time) > duration:
                    break
                try:
                    message = await asyncio.wait_for(self.websocket.recv(), timeout=0.5)
                except asyncio.TimeoutError:
                    continue
                data = json.loads(message)
                self.messages.append(data)
                if callback:
                    callback(data)
        except websockets.exceptions.ConnectionClosed:
            pass
    
    async def disconnect(self):
        """
        Disconnect from the server.
        """
        self.running = False
        if self.websocket:
            await self.websocket.close()

print("✅ ChatClient class created!")


✅ ChatClient class created!


## 🧪 Experiment: Real-time Chat!

In [4]:
# Let's test a basic WebSocket connection

async def test_connection():
    print("🔌 Testing WebSocket connection...\n")
    
    client = ChatClient("ws://localhost:5004")
    
    try:
        welcome = await client.connect("TestUser", "test-room")
        print(f"✅ Connected!")
        print(f"   Message: {welcome.get('message')}")
        print(f"   Users in room: {welcome.get('users_in_room')}")
        
        await client.disconnect()
        print("\n✅ Disconnected successfully!")
        
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        print("   Make sure the WebSocket server is running!")

asyncio.get_event_loop().run_until_complete(test_connection())

🔌 Testing WebSocket connection...

✅ Connected!
   Message: Welcome to test-room!
   Users in room: 1

✅ Disconnected successfully!


In [5]:
# Now let's have two clients chat!

async def chat_demo():
    print("💬 Two-User Chat Demo")
    print("="*50)
    
    # Create two clients
    alice = ChatClient("ws://localhost:5004")
    bob = ChatClient("ws://localhost:5004")
    
    received_messages = []
    
    def on_message(msg):
        msg_type = msg.get('type')
        if msg_type == 'message':
            print(f"   📨 {msg.get('username')}: {msg.get('text')}")
            received_messages.append(msg)
        elif msg_type == 'system':
            print(f"   🔔 {msg.get('message')}")
        elif msg_type == 'typing':
            print(f"   ✏️  {msg.get('username')} is typing...")
    
    try:
        # Alice joins
        print("\n👩 Alice connecting...")
        await alice.connect("Alice", "chat-demo")
        print("   ✅ Alice connected!")
        
        # Bob joins
        print("\n👨 Bob connecting...")
        await bob.connect("Bob", "chat-demo")
        print("   ✅ Bob connected!")
        
        # Start receiving in background for both
        alice_task = asyncio.create_task(alice.receive_messages(on_message, duration=5))
        bob_task = asyncio.create_task(bob.receive_messages(on_message, duration=5))
        
        await asyncio.sleep(0.5)  # Let them start receiving
        
        # Alice sends a message
        print("\n👩 Alice sends: 'Hi Bob!'")
        await alice.send_message("Hi Bob!")
        await asyncio.sleep(0.5)
        
        # Bob sends typing indicator then message
        print("\n👨 Bob typing...")
        await bob.send_typing()
        await asyncio.sleep(0.3)
        
        print("👨 Bob sends: 'Hey Alice! How are you?'")
        await bob.send_message("Hey Alice! How are you?")
        await asyncio.sleep(0.5)
        
        # Alice responds
        print("\n👩 Alice sends: 'Great! WebSockets are amazing!'")
        await alice.send_message("Great! WebSockets are amazing!")
        await asyncio.sleep(1)
        
        # Stop receiving
        alice.running = False
        bob.running = False
        
        await asyncio.gather(alice_task, bob_task, return_exceptions=True)
        
        print("\n" + "="*50)
        print(f"📊 Total messages exchanged: {len(received_messages)}")
        print("\n✅ This all happened over PERSISTENT connections!")
        print("   No new HTTP requests needed after initial connect!")
        
    finally:
        await alice.disconnect()
        await bob.disconnect()

asyncio.get_event_loop().run_until_complete(chat_demo())

💬 Two-User Chat Demo

👩 Alice connecting...
   ✅ Alice connected!

👨 Bob connecting...
   ✅ Bob connected!
   🔔 Bob joined the room



👩 Alice sends: 'Hi Bob!'
   📨 Alice: Hi Bob!
   📨 Alice: Hi Bob!



👨 Bob typing...
   ✏️  Bob is typing...


👨 Bob sends: 'Hey Alice! How are you?'
   📨 Bob: Hey Alice! How are you?
   📨 Bob: Hey Alice! How are you?



👩 Alice sends: 'Great! WebSockets are amazing!'
   📨 Alice: Great! WebSockets are amazing!
   📨 Alice: Great! WebSockets are amazing!



📊 Total messages exchanged: 6

✅ This all happened over PERSISTENT connections!
   No new HTTP requests needed after initial connect!


## 📊 WebSocket vs SSE: The Bidirectional Advantage

In [6]:
# Demonstrate bidirectional communication

print("📊 WebSocket vs SSE Communication")
print("="*60)
print("""
SSE (Server-Sent Events):
┌────────┐                    ┌────────┐
│ Client │ ◄──── messages ─── │ Server │
└────────┘                    └────────┘
    │                              
    └──── HTTP POST (separate connection) ─────►
    
    - One-way streaming from server
    - Client sends via separate HTTP requests
    - Two connections needed for chat!

WebSocket:
┌────────┐  ◄──── messages ───►  ┌────────┐
│ Client │  ◄──── messages ───►  │ Server │
└────────┘                       └────────┘
    
    - Full duplex - both ways!
    - Single persistent connection
    - Lower latency
    - Lower overhead
""")

📊 WebSocket vs SSE Communication

SSE (Server-Sent Events):
┌────────┐                    ┌────────┐
│ Client │ ◄──── messages ─── │ Server │
└────────┘                    └────────┘
    │                              
    └──── HTTP POST (separate connection) ─────►

    - One-way streaming from server
    - Client sends via separate HTTP requests
    - Two connections needed for chat!

WebSocket:
┌────────┐  ◄──── messages ───►  ┌────────┐
│ Client │  ◄──── messages ───►  │ Server │
└────────┘                       └────────┘

    - Full duplex - both ways!
    - Single persistent connection
    - Lower latency
    - Lower overhead



## 💓 Keeping Connections Alive: Ping / Pong Heartbeats

A WebSocket **connection can silently die** — a NAT table on your router, a corporate proxy, or a cloud load balancer will often close idle TCP connections after 30–120 seconds. Neither side is notified; the next time you try to send, you get an error (or worse, nothing).

The fix is a **heartbeat**: one side periodically sends a small ping frame and expects a pong back. If the pong doesn't arrive in time, the peer is considered dead and the connection is closed so the client can reconnect.

```
Client ──── ping ────► Server      (every ~20s)
Client ◄─── pong ───── Server      (auto-reply)
   ⏳ no pong in 10s → close + reconnect
```

The `websockets` library has this built in — just pass `ping_interval` and `ping_timeout`. Production apps (Slack, Discord, trading platforms, ...) all rely on this to detect dead peers quickly.

In [7]:
# Built-in ping/pong: the library sends a PING frame every `ping_interval`
# seconds and expects a PONG within `ping_timeout`. If no PONG arrives, the
# connection is closed automatically so the client can reconnect.

async def demo_heartbeat():
    ws = await websockets.connect(
        'ws://localhost:5004',
        ping_interval=2,   # send a PING every 2s (demo value; prod is ~20-30s)
        ping_timeout=5,    # close the connection if no PONG in 5s
    )

    # Complete the chat handshake so the server keeps the socket
    await ws.send(json.dumps({'type': 'join', 'username': 'heartbeat-bot', 'room': 'hb'}))
    await ws.recv()  # welcome

    print('🔌 Connected. Measuring round-trip with manual pings...')
    for i in range(3):
        # ws.ping() returns a Future that resolves when the PONG arrives —
        # handy for measuring real round-trip time.
        pong_waiter = await ws.ping()
        t0 = asyncio.get_event_loop().time()
        await pong_waiter
        rtt_ms = (asyncio.get_event_loop().time() - t0) * 1000
        print(f'   💓 ping {i+1}: pong received in {rtt_ms:.1f}ms')
        await asyncio.sleep(1)

    await ws.close()
    print('✅ Connection looked healthy the whole time.')

asyncio.get_event_loop().run_until_complete(demo_heartbeat())

🔌 Connected. Measuring round-trip with manual pings...
   💓 ping 1: pong received in 0.1ms


   💓 ping 2: pong received in 0.5ms


   💓 ping 3: pong received in 0.2ms


✅ Connection looked healthy the whole time.


## 🔐 Authentication & Authorization

Unlike regular HTTP requests where you attach `Authorization: Bearer <token>` to *every* call, a WebSocket is opened **once** — so you only get one shot to prove who you are. Here are the patterns you'll see in the wild, from simplest to safest:

| Pattern | How it works | Good for | Watch out |
|---------|--------------|----------|-----------|
| **Token in query string** | `ws://host/chat?token=abc` | Quick demos, internal tools | Tokens leak into server access logs & proxy logs |
| **Token in first message** | Client sends `{"type":"auth","token":"..."}` right after `connect()` | Most chat apps (Slack, Discord) | Server must close the socket on bad token — clients can keep the raw TCP open otherwise |
| **`Sec-WebSocket-Protocol` subprotocol** | Browsers allow this during the upgrade (`new WebSocket(url, ["bearer.abc..."])`) | Production web apps | Header must be echoed back by the server or the browser rejects the connection |
| **HTTP cookies on upgrade** | Same cookie your regular site uses | Same-origin web apps | Won't work cross-origin; still need CSRF-style defenses |

**Rule of thumb:** authenticate **once** during/just-after the handshake, then store the user identity server-side keyed by the socket. Never re-send the token on every chat message — that's wasteful and exposes the token to bugs that echo messages to other users.

Our lab's server uses the **"token in first message"** pattern in its simplest form — the `join` message carries the `username`. A production server would carry a JWT or session ID there and call `websocket.close(code=1008)` on a bad token. Let's demo both the happy path and what a rejection looks like.


In [ ]:
# --- Pattern 2 demo: token in first message -----------------------------
# The lab's WebSocket server accepts the very first message as a "join" frame.
# In production you would carry a JWT there; here we simulate the server's
# validation with a tiny client-side check to show the handshake shape
# without modifying the running server.

FAKE_VALID_TOKENS = {"jwt-alice-xyz", "jwt-bob-pqr"}   # normally: server-side DB / JWT check

async def authenticated_client(username: str, token: str):
    """Connect, validate token, send identity as first frame."""
    uri = "ws://localhost:5004"
    try:
        ws = await websockets.connect(uri)
    except Exception as e:
        print(f"   ❌ could not connect: {e}")
        return

    # Client-side pre-check: a real server would do this check itself and
    # call ws.close(1008) — we simulate that by refusing to send the join.
    if token not in FAKE_VALID_TOKENS:
        print(f"   🚫 [{username}] token rejected (simulating server close code=1008)")
        await ws.close()
        return

    # Happy path: first frame carries identity (+ in prod, the JWT).
    await ws.send(json.dumps({
        "type": "join",
        "username": username,
        "room": "vip",
        # "token": token,   # <- where a JWT would go in a real server
    }))
    welcome = await ws.recv()
    print(f"   ✅ [{username}] authenticated. Server said: {welcome}")
    await ws.close()

async def demo_auth():
    print("🔐 Authenticated client (valid token):")
    await authenticated_client("alice", "jwt-alice-xyz")
    print("\n🔐 Unauthenticated client (bad token):")
    await authenticated_client("mallory", "totally-forged")

asyncio.get_event_loop().run_until_complete(demo_auth())


## ⚠️ Scaling Challenges

WebSockets come with unique scaling challenges:

In [8]:
# Visualize WebSocket scaling challenges

print("⚠️ WebSocket Scaling Challenges")
print("="*60)
print("""
Challenge 1: STATEFUL CONNECTIONS
────────────────────────────────────────────────────────────
Each WebSocket connection is tied to a specific server!

┌────────┐          ┌──────────┐          ┌──────────┐
│ Alice  │◄────────►│ Server 1 │          │ Server 2 │
└────────┘          └──────────┘          └──────────┘
                                               ▲
┌────────┐                                     │
│  Bob   │◄────────────────────────────────────┘
└────────┘

Problem: To send Alice's message to Bob, Server 1 must 
         somehow tell Server 2!


Challenge 2: LOAD BALANCER CONSIDERATIONS
────────────────────────────────────────────────────────────
- L4 Load Balancers: ✅ Good! Maintains TCP connection
- L7 Load Balancers: ⚠️  Must support WebSocket upgrade
- Scaling up/down: Connections must be drained carefully


Challenge 3: DEPLOYMENTS
────────────────────────────────────────────────────────────
When you deploy new code:
- Option A: Close all connections (clients reconnect)
- Option B: Graceful handoff (complex!)

Most teams choose Option A - it's simpler!
""")

⚠️ WebSocket Scaling Challenges

Challenge 1: STATEFUL CONNECTIONS
────────────────────────────────────────────────────────────
Each WebSocket connection is tied to a specific server!

┌────────┐          ┌──────────┐          ┌──────────┐
│ Alice  │◄────────►│ Server 1 │          │ Server 2 │
└────────┘          └──────────┘          └──────────┘
                                               ▲
┌────────┐                                     │
│  Bob   │◄────────────────────────────────────┘
└────────┘

Problem: To send Alice's message to Bob, Server 1 must 
         somehow tell Server 2!


Challenge 2: LOAD BALANCER CONSIDERATIONS
────────────────────────────────────────────────────────────
- L4 Load Balancers: ✅ Good! Maintains TCP connection
- L7 Load Balancers: ⚠️  Must support WebSocket upgrade
- Scaling up/down: Connections must be drained carefully


Challenge 3: DEPLOYMENTS
────────────────────────────────────────────────────────────
When you deploy new code:
- Option A: Close

## 🏗️ Reference Architecture

For production WebSocket systems, consider this architecture:

In [9]:
# WebSocket reference architecture

print("🏗️ WebSocket Reference Architecture")
print("="*60)
print("""
                    ┌─────────────────────┐
                    │    Load Balancer    │
                    │   (L4 preferred)    │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  WebSocket  │     │  WebSocket  │     │  WebSocket  │
    │  Server 1   │     │  Server 2   │     │  Server 3   │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           └───────────────────┼───────────────────┘
                               ▼
                    ┌─────────────────────┐
                    │    Pub/Sub Layer    │
                    │  (Redis, Kafka...)  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   Backend Services  │
                    │  (stateless logic)  │
                    └─────────────────────┘

Key Points:
───────────
1. WebSocket servers are "dumb" - just handle connections
2. Pub/Sub enables cross-server communication
3. Business logic stays in stateless backend services
4. This limits the "blast radius" of WebSocket complexity!
""")

🏗️ WebSocket Reference Architecture

                    ┌─────────────────────┐
                    │    Load Balancer    │
                    │   (L4 preferred)    │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  WebSocket  │     │  WebSocket  │     │  WebSocket  │
    │  Server 1   │     │  Server 2   │     │  Server 3   │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           └───────────────────┼───────────────────┘
                               ▼
                    ┌─────────────────────┐
                    │    Pub/Sub Layer    │
                    │  (Redis, Kafka...)  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌──────

## ✅ Advantages of WebSockets

1. **Full duplex** - Both sides can send anytime
2. **Low latency** - No HTTP overhead after connection
3. **Efficient** - Small frame overhead (~2-14 bytes)
4. **Binary support** - Not just text!
5. **Wide support** - All modern browsers

## ❌ Disadvantages

1. **Stateful** - Complicates scaling and load balancing
2. **Complex infrastructure** - Need WebSocket-aware components
3. **Reconnection logic** - You must implement it
4. **Deployment challenges** - Must handle connection draining
5. **Debugging harder** - Not standard HTTP

## 🎯 When to Use WebSockets

| Use Case | Why WebSocket Works |
|----------|--------------------|
| Chat applications | Bidirectional, real-time |
| Multiplayer games | Low latency, frequent updates |
| Collaborative editing | Real-time sync both ways |
| Live trading platforms | Fast updates + user actions |
| Live auctions | Bids and updates in real-time |

### Don't use WebSocket when:

- **One-way updates only** → Use SSE instead
- **Infrequent updates** → Simple polling is fine
- **Can't handle complexity** → Start simpler

In [10]:
# Decision helper

def choose_protocol():
    print("🤔 Which Protocol Should I Use?")
    print("="*50)
    print("""
    Ask yourself these questions:
    
    1. Do I need REAL-TIME updates?
       No  → Simple Polling ✅
       Yes → Continue...
    
    2. Do I need BIDIRECTIONAL communication?
       No  → SSE ✅
       Yes → Continue...
    
    3. Are updates very frequent (>1/second)?
       No  → Long Polling might work ✅
       Yes → WebSocket ✅
    
    4. Do I need PEER-TO-PEER?
       Yes → WebRTC ✅
    """)

choose_protocol()

🤔 Which Protocol Should I Use?

    Ask yourself these questions:

    1. Do I need REAL-TIME updates?
       No  → Simple Polling ✅
       Yes → Continue...

    2. Do I need BIDIRECTIONAL communication?
       No  → SSE ✅
       Yes → Continue...

    3. Are updates very frequent (>1/second)?
       No  → Long Polling might work ✅
       Yes → WebSocket ✅

    4. Do I need PEER-TO-PEER?
       Yes → WebRTC ✅
    


## 🧪 Quick Quiz

1. **What HTTP feature allows a connection to "upgrade" to WebSocket?**

2. **You're building a notification system that only pushes alerts to users. WebSocket or SSE?**

3. **Why do WebSocket systems often use a Pub/Sub layer?**

In [11]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. HTTP 101 SWITCHING PROTOCOLS!")
print("   The Upgrade header in the request triggers this.")
print("   After 101 response, it's no longer HTTP.")
print("")
print("2. SSE! It's simpler for one-way communication.")
print("   WebSocket is overkill if users don't send data.")
print("")
print("3. CROSS-SERVER COMMUNICATION!")
print("   When Alice (on Server 1) sends a message to")
print("   Bob (on Server 2), Pub/Sub bridges them.")
print("   Without it, servers can't communicate!")

📝 Quiz Answers

1. HTTP 101 SWITCHING PROTOCOLS!
   The Upgrade header in the request triggers this.
   After 101 response, it's no longer HTTP.

2. SSE! It's simpler for one-way communication.
   WebSocket is overkill if users don't send data.

3. CROSS-SERVER COMMUNICATION!
   When Alice (on Server 1) sends a message to
   Bob (on Server 2), Pub/Sub bridges them.
   Without it, servers can't communicate!


## 📊 Final Comparison

| Feature | Polling | Long Polling | SSE | WebSocket |
|---------|---------|--------------|-----|----------|
| Latency | High | Medium | Low | Very Low |
| Direction | Request-Response | Request-Response | Server→Client | Bidirectional |
| Connection | New each time | New each response | Persistent | Persistent |
| Complexity | Very Low | Low | Medium | High |
| Scaling | Easy | Easy | Medium | Hard |
| Browser | Native | Native | EventSource | WebSocket API |

## 📚 Summary

### What We Learned:

1. **WebSocket** = Full-duplex, persistent connection
2. **HTTP upgrade** handshake establishes connection
3. **Bidirectional** - Both sides can send anytime
4. **Stateful** - Complicates scaling
5. **Use Pub/Sub** for cross-server communication
6. **Best for**: Chat, gaming, collaborative editing

### Interview Tips:

> "WebSocket is the right choice when I need true bidirectional, real-time communication. For a chat system, users need to both send and receive messages instantly. I'd use a Pub/Sub layer like Redis to handle cross-server message delivery, and keep the WebSocket servers stateless except for connection management."

### Next Up: WebRTC Overview

In the next notebook, we'll briefly cover **WebRTC** - the peer-to-peer solution for video/audio and direct client communication!